# RL Attack Path Simulation -- Google Colab Reproduction Notebook
## MMAI 845 | Syed Ali Turab

**This notebook reproduces the ENTIRE project pipeline on Google Colab -- no local setup required.**

### How to run (3 steps):
1. **Set GPU runtime:** Go to `Runtime > Change runtime type > T4 GPU`
2. **Run all cells:** Click `Runtime > Run all`
3. **Download results:** The final cell packages everything into a ZIP file

### What this notebook does (end-to-end):
| Step | Section | Time (T4 GPU) |
|------|---------|---------------|
| 1 | Setup: clone repo, install deps, verify environment | ~3 min |
| 2 | Quality gate: linting + 66 unit tests | ~5 min |
| 3 | Train PPO + DQN baseline (500k steps each) | ~15 min |
| 4 | Train PPO + DQN stealth (500k steps each) | ~15 min |
| 5 | Evaluate all 4 models (100 episodes, seed 42) | ~2 min |
| 6 | Generate figures + penetration testing report | ~1 min |
| 7 | Acceptance checks: verify expected metrics | instant |
| 8 | Package results into downloadable ZIP | instant |

**Total: ~30-40 minutes on T4 GPU.**

### Key technical details:
- **PPO:** Uses MaskablePPO (sb3-contrib) with native action masking -- invalid actions are zeroed out at the logit level before sampling
- **DQN:** Uses standard SB3 DQN with manual Q-value masking -- invalid action Q-values set to -inf before argmax
- **Environment:** Custom 5-subnet NASim topology with AI infrastructure targets (LLM server, Vector DB, Model Repo, Training Data)
- **Reward shaping:** Dense reward wrapper adds +3.0 bonus for state-changing actions; stealth wrapper adds cumulative detection + caught penalty
- **Reproducibility:** All training and evaluation uses `--seed 42` for deterministic results

---
## 1. Setup

This section checks GPU availability, clones the repository, installs all
dependencies (NASim, Stable-Baselines3, sb3-contrib for MaskablePPO, PyTorch),
and verifies the custom environment works correctly with action masking and
dense reward shaping.

In [ ]:
# =============================================================================
# Step 1a: Check GPU availability
#
# PyTorch uses CUDA for GPU-accelerated training. A T4 GPU speeds up training
# from ~60 min (CPU) to ~30 min. If no GPU is detected, the notebook will
# still work but training will be slower.
# =============================================================================
import torch

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU device:      {torch.cuda.get_device_name(0)}')
    print(f'GPU memory:      {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print('\nGPU is ready. Training will use CUDA acceleration.')
else:
    print('\nWARNING: No GPU detected!')
    print('Go to Runtime > Change runtime type > Select T4 GPU')
    print('Training will still work on CPU but will be slower (~60 min vs ~30 min).')

In [ ]:
# =============================================================================
# Step 1b: Clone the repository from GitHub
#
# This pulls the full codebase including:
#   - environments/  : Custom NASim network topology (5 subnets, 11 hosts)
#   - agents/        : PPO (MaskablePPO) and DQN agent implementations
#   - training/      : Training and evaluation CLI scripts
#   - analysis/      : Figure generation, MITRE mapping, pentest report
#   - tests/         : 66 unit tests covering all components
# =============================================================================
!git clone https://github.com/turaab97/rl-attack-path-simulation.git
%cd rl-attack-path-simulation
!echo "Repository cloned. Current directory: $(pwd)"

In [ ]:
# =============================================================================
# Step 1c: Install all dependencies
#
# 'pip install -e ".[dev]"' installs the project in editable mode with:
#   Runtime deps:  nasim, stable-baselines3, sb3-contrib (MaskablePPO),
#                  torch, gymnasium, numpy, pandas, matplotlib, seaborn
#   Dev deps:      pytest, black, isort, flake8
#
# The -q flag suppresses verbose output. This takes ~2-3 minutes.
# =============================================================================
!pip install -e ".[dev]" -q
!echo "Dependencies installed. Verifying core imports..."
!python -c "import nasim; import stable_baselines3; import sb3_contrib; print('All core packages OK')"

In [ ]:
# =============================================================================
# Step 1d: Verify the custom environment, action masking, and reward shaping
#
# This cell confirms:
#   1. The NASim environment loads with our custom 5-subnet AI infrastructure
#      topology (11 hosts, 110 discrete actions, 264-dim observation vector).
#   2. The ActionMaskWrapper correctly identifies which actions are valid
#      from the initial state (~5-15 out of 110). Invalid actions correspond
#      to hosts not yet discovered or not reachable through firewall rules.
#   3. The DenseRewardWrapper adds +3.0 bonus for state-changing actions
#      (host discovery, exploit success) to help the agent learn faster
#      than with sparse NASim rewards alone.
#
# If any of these checks fail, the environment setup is broken.
# =============================================================================
from environments.network_config import make_env, AI_INFRA_HOSTS
from agents.wrappers import IntActionWrapper, ActionMaskWrapper, DenseRewardWrapper
import numpy as np

# Create the raw NASim environment with our custom topology.
env = make_env()
obs, info = env.reset()
print(f'Environment loaded:')
print(f'  Observation shape: {obs.shape} (264 features per host)')
print(f'  Action space:      {env.action_space.n} discrete actions')
print(f'  AI targets:        {AI_INFRA_HOSTS}')
print(f'  Fully observable:  {np.count_nonzero(obs)}/{obs.shape[0]} non-zero features')

# Verify the action mask wrapper filters invalid actions.
masked_env = ActionMaskWrapper(IntActionWrapper(env))
masked_env.reset()
mask = masked_env.action_masks()
n_valid = int(mask.sum())
print(f'\nAction masking:')
print(f'  Valid actions from initial state: {n_valid}/{env.action_space.n}')
print(f'  (Only scan/exploit actions on discovered+reachable hosts are valid)')

# Verify the dense reward wrapper adds shaping bonuses.
wrapped = DenseRewardWrapper(ActionMaskWrapper(IntActionWrapper(make_env())))
obs2, _ = wrapped.reset()
_, r1, _, _, _ = wrapped.step(0)
print(f'\nDense reward wrapper:')
print(f'  Noop action reward: {r1:.2f} (expected: -1.0, no state change = no bonus)')

wrapped.close()
masked_env.close()
print('\nAll environment checks passed.')

---
## 2. Quality Gate: Linting + Unit Tests

Before training, we verify code quality and correctness:
- **black** -- checks Python formatting (PEP 8 compliance)
- **isort** -- checks import ordering
- **flake8** -- static analysis for common errors
- **pytest** -- runs 66 unit tests covering environment setup, agent training/prediction,
  evaluation harness, attack path analysis, and visualization

All tests should pass. This takes ~5 minutes.

In [ ]:
# =============================================================================
# Run linters (black, isort, flake8) and the full test suite (pytest).
#
# Linters verify code formatting and style. Tests verify:
#   - NASim environment loads correctly with our custom topology
#   - PPO and DQN agents can train, predict, save/load, and use action masks
#   - Evaluation harness produces correct metrics
#   - Attack path analysis maps actions to hosts correctly
#   - Visualization code generates plots without errors
#
# Expected: all 66 tests pass, linters report no issues.
# =============================================================================
!echo "=== Running linters ==="
!black --check .
!isort --check-only .
!flake8 .
!echo ""
!echo "=== Running test suite ==="
!pytest tests/ -v --tb=short
!echo ""
!echo "=== Quality gate passed ==="

---
## 3. Training: Baseline Mode (PPO + DQN, 500k steps each)

Trains both algorithms on the same environment for fair comparison:

- **PPO (MaskablePPO):** On-policy algorithm. Collects rollouts of experience,
  computes advantages, and updates the policy with a clipped surrogate objective.
  Action masking is native -- invalid action logits are zeroed before sampling.
  Uses entropy coefficient 0.05 for exploration in the large action space.

- **DQN (Deep Q-Network):** Off-policy algorithm. Stores transitions in a replay
  buffer and learns Q-values. Uses a target network for stability. Action masking
  is manual -- Q-values of invalid actions are set to -inf before argmax. Epsilon-
  greedy exploration samples only from valid actions.

**Reward:** Raw NASim reward (+value on host compromise, -1 per step) plus dense
reward shaping (+3.0 bonus when an action changes the observation state).

**Time:** ~15 minutes on T4 GPU for both agents combined.

In [ ]:
# =============================================================================
# Train PPO + DQN in baseline mode (no detection/stealth).
#
# --compare       : Train both PPO and DQN under identical conditions.
# --timesteps     : 500,000 environment steps per agent.
# --eval_freq     : Evaluate the agent every 25k steps during training
#                   (saves best model based on mean reward).
# --n_eval_episodes: Use 10 episodes per evaluation checkpoint.
# --seed 42       : Fixed seed for reproducible results.
#
# Output: results/ppo_baseline/ and results/dqn_baseline/ directories
#         containing final_model.zip, train_meta.json, and TensorBoard logs.
# =============================================================================
!python -m training.train --compare --timesteps 500000 --eval_freq 25000 --n_eval_episodes 10 --seed 42
!echo ""
!echo "Baseline training complete. Models saved to results/ppo_baseline/ and results/dqn_baseline/"

---
## 4. Training: Stealth Mode (PPO + DQN, 500k steps each)

Stealth mode adds a **detection model** on top of the baseline reward:

- Each active action (scan, exploit, priv-esc) adds +0.1 to a cumulative
  detection score.
- When cumulative detection reaches the threshold (0.8), the episode
  terminates with a **-100 penalty** (attacker caught by SOC/IDS).
- With detection_cost=0.1 per step, agents have at most **8 active steps**
  before being caught.

This models whether a Security Operations Centre can detect and stop an
automated attacker before it reaches AI infrastructure.

**Time:** ~15 minutes on T4 GPU for both agents combined.

In [ ]:
# =============================================================================
# Train PPO + DQN in stealth mode (SOC/IDS detection active).
#
# --stealth              : Enable the stealth reward wrapper.
# --detection_threshold  : Cumulative detection level that triggers "caught" (0.8).
# --detection_cost       : Detection added per active action (0.1 per step).
# --caught_penalty       : Terminal reward when agent is caught (-100.0).
# --alpha                : Scaling factor for detection cost in reward (1.0).
#
# Expected behaviour: agents are caught after ~8-9 active steps because
# 8 steps * 0.1 cost = 0.8 = threshold. The -100 caught penalty dominates
# the reward, producing mean_reward ~ -109.9 for both agents.
#
# Output: results/ppo_stealth/ and results/dqn_stealth/ directories.
# =============================================================================
!python -m training.train --compare --stealth --timesteps 500000 \
    --detection_threshold 0.8 --detection_cost 0.1 --caught_penalty -100.0 --alpha 1.0 \
    --eval_freq 25000 --n_eval_episodes 10 --seed 42
!echo ""
!echo "Stealth training complete. Models saved to results/ppo_stealth/ and results/dqn_stealth/"

In [ ]:
# Placeholder cell (safe no-op). Training is complete. Evaluation starts below.
print("Training phase complete. Proceeding to evaluation...")

---
## 5. Evaluation (Seeded, Reproducible)

Evaluates all 4 trained models (PPO baseline, DQN baseline, PPO stealth, DQN stealth)
over 100 episodes each with a fixed random seed (42) for deterministic results.

Metrics collected per agent:
- **mean_reward / std_reward** -- average episode return and variance
- **success_rate** -- fraction of episodes reaching all goals (+100 reward)
- **mean_steps** -- average episode length
- **catch_rate** (stealth only) -- fraction of episodes where agent was caught by SOC
- **ai_hosts_reached** -- number of AI infrastructure hosts compromised per episode

Results are saved to `results/eval_baseline.json` and `results/eval_stealth.json`.

In [ ]:
# =============================================================================
# Evaluate all 4 trained models with a fixed seed for reproducibility.
#
# Baseline evaluation:
#   Tests how well each agent navigates the network WITHOUT detection pressure.
#   Expected: PPO achieves consistent mean_reward ~ -100.0 (std 0.0);
#             DQN shows higher variance (~-498, std ~19.9).
#
# Stealth evaluation:
#   Tests how agents behave UNDER SOC/IDS detection pressure.
#   Expected: Both agents caught at step 8-9, mean_reward ~ -109.9,
#             catch_rate = 1.0 (100% caught).
#
# --episodes 100 : 100 evaluation episodes per agent for statistical reliability.
# --seed 42      : Fixed seed so results are identical across runs.
#
# Output: results/eval_baseline.json and results/eval_stealth.json
# =============================================================================
print("=" * 60)
print("  BASELINE EVALUATION (no detection)")
print("=" * 60)
!python -m training.evaluate \
    --ppo_model results/ppo_baseline/final_model \
    --dqn_model results/dqn_baseline/final_model \
    --episodes 100 --seed 42

print("\n" + "=" * 60)
print("  STEALTH EVALUATION (SOC/IDS detection active)")
print("=" * 60)
!python -m training.evaluate \
    --ppo_model results/ppo_stealth/final_model \
    --dqn_model results/dqn_stealth/final_model \
    --stealth --episodes 100 --seed 42

print("\nEvaluation complete. Results: results/eval_baseline.json, results/eval_stealth.json")

---
## 6. Generate Figures and Penetration Testing Report

Generates publication-quality matplotlib figures and an automated Markdown
penetration testing report:

- **Training curves:** reward over timesteps for each agent
- **Algorithm comparison:** bar charts of PPO vs DQN metrics
- **Network topology:** visual diagram of the 5-subnet enterprise network
- **Attack path flow:** visualization of agent traversal patterns
- **Pentest report:** MITRE ATT&CK mapping, attack path analysis, recommendations

In [ ]:
# =============================================================================
# Generate all figures and the automated penetration testing report.
#
# generate_all_figures.py: Creates training curves, comparison bar charts,
#   network topology diagram, attack path flow, and detection sensitivity plots.
#   Saved to results/plots/.
#
# report_generator: Produces a comprehensive Markdown pentest report with
#   MITRE ATT&CK technique mapping, attack path analysis, risk assessment,
#   and remediation recommendations. Saved to results/pentest_report.md.
# =============================================================================
print("Generating figures...")
!python -m analysis.generate_all_figures
print("\nGenerating penetration testing report...")
!python -m analysis.report_generator --results_dir results/ --output results/pentest_report.md
print("\nAll figures and report generated. See results/plots/ and results/pentest_report.md")

---
## 7. Acceptance Checks -- Verify Expected Results

This cell validates that the training and evaluation produced the expected
metrics. It checks:

1. **PPO baseline:** mean_reward within 20 of -100.0, std < 5.0
2. **PPO stealth:** mean_reward within 5 of -109.9, steps between 8-10, catch_rate >= 95%
3. **DQN baseline:** higher variance than PPO (expected in sparse-reward masked setting)
4. **Key artifacts:** evaluation JSONs, pentest report, and plots exist

All checks should show PASS. If a check shows CHECK, the metric is outside
the expected range (may vary slightly with different hardware/CUDA versions).

In [ ]:
# =============================================================================
# Acceptance checks: verify that evaluation metrics match expected values.
#
# These thresholds are based on multiple successful runs with seed=42.
# Small variations may occur across different GPU hardware or CUDA versions,
# which is why tolerances are included (e.g., +/- 20 for baseline reward).
# =============================================================================
import json
from pathlib import Path

results_dir = Path('results')
base_path = results_dir / 'eval_baseline.json'
stealth_path = results_dir / 'eval_stealth.json'

assert base_path.exists(), 'Missing results/eval_baseline.json -- did evaluation run?'
assert stealth_path.exists(), 'Missing results/eval_stealth.json -- did evaluation run?'

with open(base_path) as f:
    baseline = json.load(f)
with open(stealth_path) as f:
    stealth = json.load(f)

print('=== Acceptance Checks ===')

ppo_base_mean = baseline['ppo']['mean_reward']
ppo_base_std = baseline['ppo']['std_reward']
ok_ppo = abs(ppo_base_mean - (-100.0)) <= 20.0 and ppo_base_std <= 5.0
print(f"PPO baseline mean={ppo_base_mean:.2f}, std={ppo_base_std:.2f} -> {'PASS' if ok_ppo else 'CHECK'}")

ppo_stealth_mean = stealth['ppo']['mean_reward']
ppo_stealth_steps = stealth['ppo']['mean_steps']
ppo_stealth_catch = stealth['ppo']['catch_rate']
ok_stealth = abs(ppo_stealth_mean - (-109.9)) <= 5.0 and 8.0 <= ppo_stealth_steps <= 10.0 and ppo_stealth_catch >= 0.95
print(f"Stealth PPO mean={ppo_stealth_mean:.2f}, steps={ppo_stealth_steps:.1f}, catch={ppo_stealth_catch:.2f} -> {'PASS' if ok_stealth else 'CHECK'}")

print(f"DQN baseline std={baseline['dqn']['std_reward']:.2f} (higher variance vs PPO is expected in sparse masked setting)")

print('\n=== Baseline Summary ===')
for agent in ['ppo', 'dqn']:
    m = baseline[agent]
    print(f"{agent.upper()}: mean_reward={m['mean_reward']:.2f}, std={m['std_reward']:.2f}, mean_steps={m['mean_steps']:.1f}")

print('\n=== Stealth Summary ===')
for agent in ['ppo', 'dqn']:
    m = stealth[agent]
    print(f"{agent.upper()}: mean_reward={m['mean_reward']:.2f}, catch_rate={m['catch_rate']:.2f}, mean_steps={m['mean_steps']:.1f}")

print('\n=== Key Artifacts ===')
for rel in [
    'eval_baseline.json',
    'eval_stealth.json',
    'pentest_report.md',
    'plots/system_architecture.png',
    'plots/network_topology.png',
    'plots/attack_path_flow.png',
]:
    p = results_dir / rel
    print(f"{rel}: {'OK' if p.exists() else 'MISSING'}")

---
## 8. Download Results

Packages the entire `results/` directory into a ZIP file and triggers a browser
download. The ZIP contains:

- `eval_baseline.json` / `eval_stealth.json` -- evaluation metrics
- `ppo_baseline/` / `dqn_baseline/` -- trained baseline models + training metadata
- `ppo_stealth/` / `dqn_stealth/` -- trained stealth models + training metadata
- `plots/` -- all generated figures (PNG)
- `pentest_report.md` -- automated penetration testing report

To use these results locally, unzip and copy the `results/` folder into your
local clone of the repository.

In [ ]:
# =============================================================================
# Package all results into a ZIP and trigger browser download.
#
# The ZIP file (~50-100 MB) contains trained models, evaluation metrics,
# figures, and the pentest report. Save it locally and unzip into
# your repo clone's results/ directory for further analysis.
# =============================================================================
!zip -r /content/rl_results.zip results/
print(f"\nZIP created: /content/rl_results.zip")
!ls -lh /content/rl_results.zip

from google.colab import files
files.download('/content/rl_results.zip')
print('\nDownload triggered. Save the file and unzip into your local repo clone.')

---

*Training notebook by Syed Ali Turab -- MMAI 845, Queen's University*